# TAAF deploy workflow

A walkthrough of `taaf.deploy` / `taaf.deploy_inline` / `taaf.deploy_slurm` as currently implemented:

- `Benchmark.deploy(target)` (R2.21) — the polymorphic entry point
- `InlineTarget` — run in the current process (R2.22)
- `TufaSlurmTarget` — submit as a Slurm job, get a handle back without blocking (R2.23, R2.37, R2.38)
- `handle.wait()` / `stop()` / `is_done` — controlling a live job (R2.33)
- `DeploymentHandle.attach(job_dir)` — reattach to a previously-launched job from a fresh Python session (R2.39)

The Slurm cells call `sbatch` for real — run this notebook from a Slurm submit host (`ssh tufaslurm` or equivalent). Outputs land under `notebooks/runs/deploy/` (gitignored).

Spec: [`docs/requirements.md`](../docs/requirements.md) — status: [`docs/requirement_status.md`](../docs/requirement_status.md).

In [ ]:
import json
from pathlib import Path

import arcengine

import taaf

OUT = Path("runs/deploy")
OUT.mkdir(parents=True, exist_ok=True)


def _act(action_id: int) -> arcengine.ActionInput:
    return arcengine.ActionInput(id=arcengine.GameAction.from_id(action_id), data={})


def make_benchmark(label: str, job_dir: Path) -> taaf.benchmark.Benchmark:
    """Deterministic single-game / single-pass benchmark that wins ExampleGame."""
    return taaf.benchmark.Benchmark(
        label=label,
        games=[taaf.game_examples.ExampleGame()],
        solver=taaf.solver_examples.SolverSequence(actions=[_act(1)] * 3 + [_act(2)] * 3),
        n_passes=1,
        job_dir=job_dir,
    )

## InlineTarget (R2.22)

The simplest deployment: `await bm.deploy(InlineTarget())` runs in the current process and blocks the awaiting task until the benchmark finishes. The returned handle is mostly a typed receipt — `is_done` is already `True`, `wait()` returns the populated benchmark, `stop()` is a no-op.

Notebook cells run in an async context, so `await` works directly. In a script, wrap with `asyncio.run(bm.deploy(...))`.

Use this for smoke tests on a laptop or inside an interactive Slurm allocation.

In [ ]:
inline_job_dir = OUT / "inline"
bm = make_benchmark("notebook_inline", inline_job_dir)
inline_handle = await bm.deploy(taaf.deploy_inline.InlineTarget())

print("handle.is_done :", inline_handle.is_done)
print("handle.job_dir :", inline_handle.job_dir)
print("game state     :", bm.game_runs[0].state, "score:", bm.game_runs[0].final_score)

What InlineTarget wrote into `job_dir`:

In [ ]:
for p in sorted(inline_job_dir.rglob("*")):
    if p.is_file() and ".venv" not in p.parts:
        print(f"  {p.relative_to(inline_job_dir)}  ({p.stat().st_size} bytes)")

## TufaSlurmTarget (R2.23)

Submits the benchmark as a Slurm job via `sbatch --parsable` and returns immediately — the run happens asynchronously on a worker node.

The launcher writes:

- `benchmark_initial.pkl` — the worker unpickles this to get the benchmark
- `run_in_worker.py` — the entry script `python run_in_worker.py` invokes
- `sbatch_script.sh` — the rendered sbatch script (for grep / debug)
- `deploy_meta.json` — what `attach()` reads to rebuild a handle later
- `src/<repo>/` — source snapshot (R2.35) installed editable into a per-run venv (R2.37) inside the sbatch script

GPU type maps to `--gres=gpu:<type>:1`. Optional `image` wraps the work in `srun --container-image=… bash -c '…'` (R2.38).

In [ ]:
slurm_job_dir = OUT / "slurm"
bm = make_benchmark("notebook_slurm", slurm_job_dir)

slurm_handle = await bm.deploy(taaf.deploy_slurm.TufaSlurmTarget(
    gpu="B200",
    time="00:30:00",
    image="nvcr.io/nvidia/pytorch:25.01-py3",
    partition="llm",
))

print("job_id :", slurm_handle.job_id)
print("job_dir:", slurm_handle.job_dir)

Inspect what `deploy()` rendered — the sbatch script the cluster will run:

In [ ]:
print((slurm_job_dir / "sbatch_script.sh").read_text())

## handle.wait() / stop() / is_done (R2.33)

`is_done` is a single `squeue -j <id>` query — empty output ⇒ job has left the queue ⇒ done.

In [ ]:
print("is_done:", slurm_handle.is_done)

`wait()` polls `squeue` every 30s by default and returns the populated `Benchmark` once the job leaves the queue:

```python
result = slurm_handle.wait()
print(result.game_runs[0].state, result.game_runs[0].final_score)
```

For a real run this could block for the full walltime. Usually you'd skip `wait()` here and reattach later (next section); calling it makes sense in a launcher script that's expected to babysit the job.

To cancel:

```python
slurm_handle.stop()   # scancel handle.job_id; idempotent — safe even if already done
```

## DeploymentHandle.attach(job_dir) (R2.39)

If your launcher Python died but the Slurm job kept running, you can rebuild the handle from disk. Works uniformly across targets via `job_dir/deploy_meta.json` — `attach()` reads the recorded `handle_class`, imports it, and dispatches to its `_attach()` classmethod.

- **Inline**: any extant `deploy_meta.json` describes a finished run (inline can't outlive its Python process), so `_attach` reloads the benchmark pickle eagerly.
- **Slurm**: `_attach` only carries the `job_id`; loading the benchmark is deferred to `wait()`, which may still need to poll.

In [ ]:
inline_reattached = taaf.deploy.DeploymentHandle.attach(inline_job_dir)
print("inline:", type(inline_reattached).__name__,
      "is_done:", inline_reattached.is_done,
      "game:", inline_reattached.wait().game_runs[0].state)

slurm_reattached = taaf.deploy.DeploymentHandle.attach(slurm_job_dir)
print("slurm :", type(slurm_reattached).__name__,
      "job_id:", slurm_reattached.job_id)

The metadata file itself — handy for shell one-liners like `jq -r .job_id deploy_meta.json | xargs scancel`:

In [ ]:
print(json.dumps(json.loads((slurm_job_dir / "deploy_meta.json").read_text()), indent=2))

## Where things land

| File in `job_dir`                  | Written by                            | Purpose                                                              |
|------------------------------------|---------------------------------------|----------------------------------------------------------------------|
| `benchmark.json`                   | `Benchmark.run` (R2.13 / R2.14)       | Pure-JSON results snapshot                                          |
| `benchmark.pkl`                    | `Benchmark.run` teardown              | Full pickle — reload via `Benchmark.from_pickle`                    |
| `benchmark_initial.pkl`            | `TufaSlurmTarget.deploy`              | Pre-run pickle the worker unpickles to bootstrap                    |
| `deploy_target.pkl`                | `TufaSlurmTarget.deploy`              | Pickled target so the worker can pass it as `runtime_environment=` into `bm.run` (R12.01) |
| `diagnostics.html` / `summary.txt` | `taaf.diagnostics` (R4.01 / R4.02)    | Per-run reports                                                     |
| `stdout.log` / `stderr.log`        | tee (Inline) / sbatch (Slurm)         | Captured stdout+stderr (R2.36)                                      |
| `src/<repo>/...`                   | `snapshot_editable_sources` (R2.35)   | Source bundle for the run — every editable repo in the launcher's venv |
| `deploy_meta.json`                 | `write_deploy_meta` (R2.39)           | What `DeploymentHandle.attach(job_dir)` reads                       |
| `run_in_worker.py`                 | `TufaSlurmTarget.deploy`              | Worker entry script                                                 |
| `sbatch_script.sh`                 | `TufaSlurmTarget.deploy`              | Generated sbatch script (debug-friendly)                            |

### Source

- [`src/taaf/deploy.py`](../src/taaf/deploy.py) — `DeploymentTarget` / `DeploymentHandle` ABCs, `attach()`, metadata, source/preamble helpers
- [`src/taaf/deploy_inline.py`](../src/taaf/deploy_inline.py) — `InlineTarget`
- [`src/taaf/deploy_slurm.py`](../src/taaf/deploy_slurm.py) — `TufaSlurmTarget`
- [`docs/requirements.md`](../docs/requirements.md) — R2.21–R2.39 spec
- [`docs/requirement_status.md`](../docs/requirement_status.md) — what's implemented vs deferred